In [1]:
import pandas as pd
import math
!pip install pyvis
import nltk
from pyvis.network import Network
from IPython.display import display, HTML

from nltk.sentiment.vader import SentimentIntensityAnalyzer



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.1 MB/s eta 0:00:00


In [4]:
#if you have access to the dirve files run this cell and skip the second
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
file_path = '/content/drive/MyDrive/DSAFinalProject/DSAtranscription.csv'
df = pd.read_csv(file_path)

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#if you dont have access to dirve run this cell if you do ignore

dummy_data = [
        {'start_time': 1.0, 'speaker_id': 'Player 1', 'text': 'Hey, look at this lock!'},
        {'start_time': 3.0, 'speaker_id': 'Player 2', 'text': 'I see it. Is there a clue?'},
        {'start_time': 5.0, 'speaker_id': 'Player 1', 'text': 'Yes, grab the flashlight.'}
    ]
df = pd.DataFrame(dummy_data)
file_path = None

In [5]:

nltk.download('vader_lexicon', quiet=True)
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
  return analyzer.polarity_scores(text)['compound']

In [6]:
#Added sentiment-based edge coloring to the graph.
#The average sentiment score for each edge determines its color:
#[-1, -0.3] = negative interaction, [-0.3, 0.3] = neutral, above 0.3 = positive.
#Can be reverted to the previous version if not needed.




#making a doubly linked list (sliding window)
class DLLNode:
  """
  A node in doubly linked list, represents a single spoken event or utterance
  """
  def __init__(self, time, speaker, sentiment):
    """
    Intitializes new node

    arguments:
            time (float): timestamp of spoken even
            speaker (str): the indetifer of person speakeing
            sentiment (float): vader sentment score of spoken text -1.0 to 1.0
    """
    self.time = time
    self.speaker = speaker
    self.sentiment = sentiment
    self.next = None
    self.prev = None


class TemporalWindowList:
  """
  custom doubly linked list class that maintains a sliding temporal window of events
  events that fall outsilde the specified time window are removed from the front
  """
  def __init__(self, window_seconds=5.0):
    """
      Intializes the TemporalwIndowList

      Args:
        window_seconds (float): The maximum age (in seconds) of events to keep in the sliding window Default is 5.0
    """
    self.head = None
    self.tail = None
    self.window_seconds = window_seconds


  def append(self, time, speaker, sentiment):
    """
    Adds a new spoken event to the tail of the doubly linked list in O(1) time.

        Args:
            time (float): The timestamp of the new event.
            speaker (str): The identifier of the person speaking.
            sentiment (float): The sentiment score of the spoken text.

    """
    new_node = DLLNode(time, speaker, sentiment)
    if self.tail is None:
      self.head = self.tail = new_node
    else:
      self.tail.next = new_node
      new_node.prev = self.tail
      self.tail = new_node


  def remove_old_events(self, current_time):
      """Removes events from the front of the list that fall outside the time window.
          Input: current_time (float)
          output: nothing returned
      """
      while self.head and (current_time - self.head.time) > self.window_seconds:
        self.head = self.head.next
        if self.head:
          self.head.prev = None
        else:
          self.tail = None


  def get_recent_speakers(self):
    """Iterates through current window and finds who recently spoke,
        input: none Output: A list of the recent speakers who are currently in the window
    """
    speakers = []
    current = self.head
    while current:
      speakers.append(current.speaker)
      current = current.next
    return speakers



#Graph implentation
class InteractionGraph:
  """
    A directed, weighted graph representing interactions between speakers.
    Implemented using an adjacency list.
    Also tracks sentiment scores for each edge.
    """
  def __init__(self):
    """
        Initializes the InteractionGraph with an empty adjacency list.
        The goal is to map speaker -> {addressee: weight}.
        Also initializes a sentiment tracker for each edge.

        No args
    """
    self.vertices = {}
    self.sentiments = {}

  def add_interaction(self, speaker, addressee, sentiment=0.0):
    """
        Adds a directed interaction (edge) from a speaker to an addressee.
        Increments the weight if the edge already exists. Ignores self-loops.
        Stores the sentiment score for averaging later.

        Args:
            speaker (str): The identifier of the person initiating the interaction.
            addressee (str): The identifier of the person receiving the interaction.
            sentiment (float): The sentiment score of the utterance (-1.0 to 1.0).
        """

    if speaker == addressee:
      return
    if speaker not in self.vertices:
      self.vertices[speaker] = {}
    if addressee not in self.vertices:
      self.vertices[addressee] = {}

    if addressee not in self.vertices[speaker]:
      self.vertices[speaker][addressee] = 0

    self.vertices[speaker][addressee] += 1

    key = (speaker, addressee)
    if key not in self.sentiments:
      self.sentiments[key] = []
    self.sentiments[key].append(sentiment)

  def get_avg_sentiment(self, speaker, addressee):
    """
        Returns the average sentiment score for a specific edge.

        Args:
            speaker (str): The source node.
            addressee (str): The target node.

        Returns:
            float: Average sentiment score for that edge. 0.0 if no data.
        """
    key = (speaker, addressee)
    if key not in self.sentiments or len(self.sentiments[key]) == 0:
      return 0.0
    return sum(self.sentiments[key]) / len(self.sentiments[key])

  def get_out_degree(self, player):
    """
        Calculates the total number of outbound interactions for a given player.

        Args:
            player (str): The identifier of the player.

        Returns:
            int: The total weight of all edges originating from the specified player.
                 Returns 0 if the player is not found in the graph.
        """
    if player not in self.vertices:
      return 0
    return sum(self.vertices[player].values())

In [7]:

import unittest

class TestDLLNode(unittest.TestCase):
    """Tests for the doubly linked list node"""

    def test_node_creation(self):
        node = DLLNode(1.0, "Player 1", 0.5)
        self.assertEqual(node.time, 1.0)
        self.assertEqual(node.speaker, "Player 1")
        self.assertEqual(node.sentiment, 0.5)
        self.assertIsNone(node.next)
        self.assertIsNone(node.prev)

    def test_node_linking(self):
        node1 = DLLNode(1.0, "Player 1", 0.5)
        node2 = DLLNode(2.0, "Player 2", -0.3)
        node1.next = node2
        node2.prev = node1
        self.assertEqual(node1.next, node2)
        self.assertEqual(node2.prev, node1)


class TestTemporalWindowList(unittest.TestCase):
    """Tests for the sliding temporal window (doubly linked list)"""

    def test_append_to_empty_list(self):
        window = TemporalWindowList(window_seconds=5.0)
        window.append(1.0, "Player 1", 0.5)
        self.assertIsNotNone(window.head)
        self.assertIsNotNone(window.tail)
        self.assertEqual(window.head.speaker, "Player 1")
        self.assertEqual(window.head, window.tail)

    def test_append_multiple(self):
        window = TemporalWindowList(window_seconds=5.0)
        window.append(1.0, "Player 1", 0.5)
        window.append(2.0, "Player 2", -0.3)
        window.append(3.0, "Player 3", 0.0)
        self.assertEqual(window.head.speaker, "Player 1")
        self.assertEqual(window.tail.speaker, "Player 3")
        self.assertEqual(window.head.next.speaker, "Player 2")

    def test_remove_old_events(self):
        window = TemporalWindowList(window_seconds=4.0)
        window.append(1.0, "Player 1", 0.5)
        window.append(3.0, "Player 2", -0.3)
        window.append(6.0, "Player 3", 0.0)
        window.remove_old_events(6.0)
        speakers = window.get_recent_speakers()
        self.assertNotIn("Player 1", speakers)
        self.assertIn("Player 2", speakers)
        self.assertIn("Player 3", speakers)

    def test_remove_all_events(self):
        window = TemporalWindowList(window_seconds=2.0)
        window.append(1.0, "Player 1", 0.5)
        window.append(2.0, "Player 2", -0.3)
        window.remove_old_events(100.0)
        self.assertIsNone(window.head)
        self.assertIsNone(window.tail)
        self.assertEqual(window.get_recent_speakers(), [])

    def test_get_recent_speakers_empty(self):
        window = TemporalWindowList(window_seconds=5.0)
        self.assertEqual(window.get_recent_speakers(), [])

    def test_window_boundary(self):
        window = TemporalWindowList(window_seconds=4.0)
        window.append(1.0, "Player 1", 0.5)
        window.remove_old_events(5.0)
        self.assertEqual(window.get_recent_speakers(), ["Player 1"])
        window.remove_old_events(5.01)
        self.assertEqual(window.get_recent_speakers(), [])


class TestInteractionGraph(unittest.TestCase):
    """Tests for the directed weighted interaction graph"""

    def test_add_single_interaction(self):
        graph = InteractionGraph()
        graph.add_interaction("Player 1", "Player 2")
        self.assertEqual(graph.vertices["Player 1"]["Player 2"], 1)

    def test_add_multiple_interactions(self):
        graph = InteractionGraph()
        graph.add_interaction("Player 1", "Player 2")
        graph.add_interaction("Player 1", "Player 2")
        graph.add_interaction("Player 1", "Player 2")
        self.assertEqual(graph.vertices["Player 1"]["Player 2"], 3)

    def test_ignore_self_loop(self):
        graph = InteractionGraph()
        graph.add_interaction("Player 1", "Player 1")
        self.assertEqual(graph.vertices, {})

    def test_directed_edges(self):
        graph = InteractionGraph()
        graph.add_interaction("Player 1", "Player 2")
        self.assertIn("Player 2", graph.vertices["Player 1"])
        self.assertNotIn("Player 1", graph.vertices.get("Player 2", {}))

    def test_get_out_degree(self):
        graph = InteractionGraph()
        graph.add_interaction("Player 1", "Player 2")
        graph.add_interaction("Player 1", "Player 2")
        graph.add_interaction("Player 1", "Player 3")
        self.assertEqual(graph.get_out_degree("Player 1"), 3)



    def test_empty_graph(self):
        graph = InteractionGraph()
        self.assertEqual(graph.vertices, {})
        self.assertEqual(graph.get_out_degree("Player 1"), 0)


unittest.main(argv=[''], exit=False, verbosity=2)

test_node_creation (__main__.TestDLLNode.test_node_creation) ... ok
test_node_linking (__main__.TestDLLNode.test_node_linking) ... ok
test_add_multiple_interactions (__main__.TestInteractionGraph.test_add_multiple_interactions) ... ok
test_add_single_interaction (__main__.TestInteractionGraph.test_add_single_interaction) ... ok
test_directed_edges (__main__.TestInteractionGraph.test_directed_edges) ... ok
test_empty_graph (__main__.TestInteractionGraph.test_empty_graph) ... ok
test_get_out_degree (__main__.TestInteractionGraph.test_get_out_degree) ... ok
test_ignore_self_loop (__main__.TestInteractionGraph.test_ignore_self_loop) ... ok
test_append_multiple (__main__.TestTemporalWindowList.test_append_multiple) ... ok
test_append_to_empty_list (__main__.TestTemporalWindowList.test_append_to_empty_list) ... ok
test_get_recent_speakers_empty (__main__.TestTemporalWindowList.test_get_recent_speakers_empty) ... ok
test_remove_all_events (__main__.TestTemporalWindowList.test_remove_all_event

In [15]:


# Added sentiment-based edge coloring to the graph.
# The average sentiment score for each edge determines its color:
# [-1, -0.3] = negative interaction, [-0.3, 0.3] = neutral, above 0.3 = positive.
# Can be reverted to the previous version if not needed.


#Definging time window
time_window = TemporalWindowList(window_seconds = 4.0)
team_graph = InteractionGraph()

class spoken_at_queue:
  def __init__(self):
    self.items = []

  def enqueue(self, item):
    self.items.append(item)

  def query_by_time(self, target_time, window=2.0):
    results = []

    for event in self.items:
      event_time, speaker, text = event

      if abs(event_time - target_time) <= window:
        results.append(event)

    return results

transcript_queue = spoken_at_queue()

print("Processing trancscript and building graph...")
for index, row in df.iterrows():
  speaker = str(row.get('speaker_id', 'Unknown').strip())
  text = str(row.get('text', ''))

  try:
    start_time = float(row.get('start_time', 0))
  except:
    continue

  if pd.isna(text) or text.strip() == "" or speaker == 'Unknown':
    continue

  transcript_queue.enqueue((start_time, speaker, text))


  #move window forward
  time_window.remove_old_events(start_time)

  #whoever is still in window is "replied" to by current speaker
  recent_speakers = time_window.get_recent_speakers()
  sentiment = get_sentiment(text)
  for prev_speaker in recent_speakers:
    #add directed edge from current spekaer to preveious speaker
    team_graph.add_interaction(speaker, prev_speaker, sentiment)

  #add curr speaker to window for future replies
  time_window.append(start_time, speaker, sentiment)

def get_in_degree(graph, player):
  """Counts how many interactions are directed towards this player."""
  total = 0
  for speaker, connections in graph.vertices.items():
    total += connections.get(player, 0)
  return total

def classify_role(graph, player):
    """Classifies each player based on relative communication behavior."""
    players = list(graph.vertices.keys())
    if not players:
        return "Unknown"

    # 1. Calculate group averages
    total_out = sum(graph.get_out_degree(p) for p in players)
    total_in = sum(get_in_degree(graph, p) for p in players)

    avg_out = total_out / len(players) if players else 0
    avg_in = total_in / len(players) if players else 0

    # 2. Get player stats
    out_degree = graph.get_out_degree(player)
    in_degree = get_in_degree(graph, player)

    # 3. Classify based on relation to the average
    if out_degree >= avg_out and in_degree >= avg_in:
        return "Leader / Coordinator"
    elif out_degree >= avg_out and in_degree < avg_in:
        return "Active Speaker"
    elif out_degree < avg_out and in_degree >= avg_in:
        return "Frequently Addressed"
    else:
        return "Observer / Quiet Participant"

for player in team_graph.vertices:
  role = classify_role(team_graph, player)
  print(player, ":", role)


# Gnerarting the graph part
print("Rendering...")
net = Network(notebook = True, directed = True, width = "100%", height = "600px", bgcolor = "#222222", font_color ="white", cdn_resources='remote')
net.set_edge_smooth('dynamic')
net.repulsion(node_distance = 500, central_gravity = 0.05, spring_length = 250)

#colors
player_colors = {
    "Player 1": "#FF8C42",
    "Player 2": "#43AA8B",
    "Player 3": "#4A90E2",
    "Player 4": "#AF7AC5",

}
default_color = "#E5E5E5"


def get_edge_color(avg_sentiment):
  """Returns edge color based on average sentiment score"""
  if avg_sentiment < -0.3:
    return "#FF4444"
  elif avg_sentiment > 0.3:
    return "#44BB44"
  else:
    return "#888888"


#nodes
players = list(team_graph.vertices.keys())
for player in players:
  node_color = player_colors.get(player, default_color)
  activity_level = team_graph.get_out_degree(player)
  net.add_node(player,
               label=player,
               size=10 + (activity_level * 0.5),
               color = node_color,
               title =f"Total Replies: {activity_level}")

#drawing edges calcuated by graph class
for speaker, connections in team_graph.vertices.items():
  for addressee, weight in connections.items():
    if weight > 0:
      avg_sent = team_graph.get_avg_sentiment(speaker, addressee)
      edge_color = get_edge_color(avg_sent)
      net.add_edge(speaker, addressee, value=weight,
                   title=f"{weight} interactions (sentiment: {avg_sent:.2f})",
                   color=edge_color)

#forcing collab to run it directily
net.write_html("interaction_network.html")
display(HTML(filename ="interaction_network.html"))


Processing trancscript and building graph...
Player 1 : Leader / Coordinator
Player 3 : Observer / Quiet Participant
Player 4 : Leader / Coordinator
Player 2 : Observer / Quiet Participant
Rendering...


In [ ]:
"""This is the transcript query tool. Type in a number and it will return the transcript
   around that second interval."""

while True:
  user_input = input("\nEnter a time in seconds to search transcript (or type 'exit'): ")

  if user_input.lower() == "exit":
    print("Exiting transcript query tool.")
    break

  try:
    target_time = float(user_input)
  except:
    print("Invalid input. Please enter a number like 10 or 15.5")
    continue

  results = transcript_queue.query_by_time(target_time, window=2.0)

  if len(results) == 0:
    print(f"No transcript events found near {target_time} seconds.")
  else:
    print(f"\nTranscript events near {target_time} seconds:")
    for event in results:
      event_time, speaker, text = event
      print(f"[{event_time:.1f}s] {speaker}: {text}")


Transcript events near 1.0 seconds:
[1.0s] Player 1: Hey, look at this lock!
[3.0s] Player 2: I see it. Is there a clue?

Transcript events near 3.0 seconds:
[1.0s] Player 1: Hey, look at this lock!
[3.0s] Player 2: I see it. Is there a clue?
[5.0s] Player 1: Yes, grab the flashlight.
